# RealBackhaulNet: real-data training and frozen evaluation

This notebook reads the immutable evidence produced by the A100 run. It does not generate rows, replace missing labels, or join records from unrelated datasets. Every example comes from a processed public-source split whose raw and processed SHA-256 values are recorded in the manifests.

In [1]:
from pathlib import Path
import hashlib, json, os
import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'real_policy').is_dir())
EVIDENCE = ROOT / 'real_policy' / 'evidence'
ARTIFACTS = ROOT / 'real_policy' / 'submission' / 'artifacts'
PROCESSED = Path(os.environ.get('REAL_DATA_ROOT', ROOT / 'data_sources' / 'processed'))
assert PROCESSED.exists(), 'set REAL_DATA_ROOT to the prepared real-data directory'

data_manifest = json.loads((EVIDENCE / 'data_manifest.json').read_text())
metrics = json.loads((EVIDENCE / 'metrics.json').read_text())
model_manifest = json.loads((ARTIFACTS / 'manifest.json').read_text())
assert data_manifest['contains_synthetic'] is False
assert data_manifest['cross_source_rows_joined'] is False
assert data_manifest['candidate_selection_reads_target'] is False
assert model_manifest['pretrained_weights'] is False
assert model_manifest['official_final_opened_after_freeze'] is True
print({'torch': torch.__version__, 'processed_data_available': PROCESSED.exists(), 'artifact': model_manifest['artifact']['path']})

{'torch': '1.13.1+cu116', 'processed_data_available': True, 'artifact': 'real_backhaul_policy_frozen.pt'}


## Exact processed splits

Rows below are model examples, not necessarily raw-source rows. Route sources produce one example per eligible observed transition; other sources produce trips, tracks, trucks, sensor records, or taxi trips.

In [2]:
split_rows = []
for source, entry in data_manifest['sources'].items():
    for split, file_info in entry['files'].items():
        split_rows.append({
            'source': source, 'split': split, 'examples': file_info['rows'],
            'bytes': file_info['bytes'], 'sha256': file_info['sha256'][:16] + '...',
        })
split_table = pd.DataFrame(split_rows).sort_values(['source', 'split']).reset_index(drop=True)
display(split_table)
print('total processed examples:', int(split_table.examples.sum()))

,source,split,examples,bytes,sha256
0,amazon,final,10000,10648002,468573559cfac378...
1,amazon,train,60000,63642242,7ebaef8fc5df8903...
2,amazon,val,10000,10589447,d6260e3faafe8e06...
3,dtcargo,test,16336,679876,689ae9ec5249ce6c...
4,dtcargo,train,71120,2954565,a1aa34ad7cd270e7...
5,dtcargo,val,14370,615874,645d0ac9eac95818...
6,lade,test,10000,2799652,917fa61644a7362d...
7,lade,train,60000,16716723,427b48296a0040fd...
8,lade,val,10000,2784499,34fc55a581753496...
9,scania,final,16000,5787220,c88c96ade2f9ebdb...


total processed examples: 504996


## One-time held-out results

Final partitions were first opened only after validation selected the checkpoint and the weights were frozen. The deadline-bounded release evaluates at most 24 batches (4,608 rows) per source and records each exact count. Canonical regression metrics are in each head's transformed source space; the paper and model card label them explicitly instead of presenting them as Haulio business KPIs.

In [3]:
metric_rows = []
for source, values in metrics['final_held_out'].items():
    for name, value in values.items():
        if name != 'examples':
            metric_rows.append({'source': source, 'metric': name, 'value': value})
heldout = pd.DataFrame(metric_rows).sort_values(['source', 'metric']).reset_index(drop=True)
display(heldout)
print({'best_step': metrics['selection']['best_step'], 'wall_seconds': metrics['wall_seconds'], 'completed_steps': metrics['completed_steps']})

,source,metric,value
0,amazon,hard_constraint_violations,0.000000
1,amazon,nearest_neighbor_top1,0.620443
2,amazon,next_stop_top1,0.693359
3,amazon,next_stop_top3,0.913411
4,dtcargo,duration_mae_canonical,0.625585
5,dtcargo,home_base_average_precision,0.709232
6,dtcargo,long_haul_average_precision,0.159211
7,dtcargo,signal_mae_canonical,0.339071
8,health,aps_failure_average_precision,0.747407
9,health,aps_failure_recall_at_0_5,0.929825


{'best_step': 800, 'wall_seconds': 2433.1051478385925, 'completed_steps': 1100}


## Frozen-policy replay on real held-out route transitions

This cell loads transformed Amazon and LaDe final examples directly. It verifies checksum binding, finite output, hard-mask compliance, and next-stop metrics without constructing any Haulio-shaped fake episode.

In [4]:
artifact_path = ARTIFACTS / model_manifest['artifact']['path']
digest = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
assert digest == model_manifest['artifact']['sha256']
frozen = torch.jit.load(str(artifact_path), map_location='cpu').eval()
replay = []
for source, filename in [('amazon', 'amazon_final.npz'), ('lade', 'lade_test.npz')]:
    ds = np.load(PROCESSED / filename, allow_pickle=False)
    n = min(32, len(ds['target']))
    nodes = torch.from_numpy(ds['nodes'][:n]).float()
    context = torch.from_numpy(ds['context'][:n]).float()
    mask = torch.from_numpy(ds['mask'][:n]).bool()
    feasible = torch.from_numpy(ds['feasible'][:n]).bool() if 'feasible' in ds.files else mask
    target = torch.from_numpy(ds['target'][:n]).long()
    with torch.inference_mode():
        logits, eta, stop = frozen(nodes, context, mask, feasible)
    prediction = logits.argmax(1)
    nearest = nodes[:, :, 2].masked_fill(~(mask & feasible), 1e6).argmin(1)
    replay.append({
        'source': source, 'real_examples': n,
        'pointer_top1': float((prediction == target).float().mean()),
        'nearest_top1': float((nearest == target).float().mean()),
        'nonfinite_outputs': int((~torch.isfinite(logits)).sum() + (~torch.isfinite(eta)).sum() + (~torch.isfinite(stop)).sum()),
        'masked_selection_violations': int((~(mask & feasible)[torch.arange(n), prediction]).sum()),
    })
display(pd.DataFrame(replay))

,source,real_examples,pointer_top1,nearest_top1,nonfinite_outputs,masked_selection_violations
0,amazon,32,0.81250,0.71875,0,0
1,lade,32,0.53125,0.31250,0,0


## Leakage and release gates

The final assertions are designed to fail loudly if someone swaps in a synthetic manifest, joins source rows, exposes mutable training behavior, or changes the frozen artifact without updating its checksum.

In [5]:
assert model_manifest['random_initialization'] is True
assert model_manifest['contains_synthetic'] is False
assert model_manifest['cross_dataset_rows_joined'] is False
assert model_manifest['runtime_auto_update'] is False
assert model_manifest['dispatcher_approval_required'] is True
assert metrics['artifact']['max_abs_parity_difference'] <= 1e-5
assert all(row['masked_selection_violations'] == 0 for row in replay)
print('PASS: real-source lineage, frozen parity, mask safety, and immutable-runtime gates')

PASS: real-source lineage, frozen parity, mask safety, and immutable-runtime gates


## Interpretation

This is an actual trained neural product artifact, but its evidence remains component-level and cross-domain. It validates learned route-sequence, IoT, heavy-truck, annual deadhead, APS-health, and cost-proxy tasks on real held-out public observations. It does **not** validate an end-to-end Haulio empty-kilometre or margin improvement, because no public source contains that aligned outcome. Missing operational facts therefore trigger deterministic constraints or `ABSTAIN`, never fabricated values.